# Material-fitted 3D local meshes and nonuniform face partitions

An exact oblique-interface transmission problem verifies physical area weights, tensor diffusion, mixed data and one-sided RT reconstruction. The archived nine patches are not a convergence-rate study.

In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
from functools import partial
from threadpoolctl import threadpool_limits
from IPython.display import Image, display
from pymhm.meshes.tetrahedron import TetraMesh
from pymhm.fem.traces.triangle_3d import TriangularSkeleton
from pymhm.meshes.fitting import fit_planar_material, planar_face_partitions
from pymhm.recovery.moments_3d import reconstruct_darcy_moments_3d

sys.path.insert(0, str(ROOT / "examples"))
from planar3d_data import Planar3DData


## Declare local and global forms

The application defines the energy/source and both oriented trace incidences.
Qk Cartesian and Pk tetrahedral coordinates use their shared FEM kernels:

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T, & L_T(v)&=(f,v)_T,\\
b_T(\lambda,v)&=\langle\lambda,v\rangle_{\partial T}, &
c_T(p,\mu)&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

The declared `LocalEquations` use C=-B.T, a constant pressure kernel and its
physical integral moment. The additional global `Equation` supplies the weak
Dirichlet pressure functional with its prescribed sign. Neumann data are
outward physical fluxes; a fully Neumann boundary uses an explicit pressure
integral constraint. Generic assembly and field interpretation are separate
calls. The sampled gradient flux remains distinct from the conservative trace.


In [ ]:
from pymhm import assemble
from examples.formulations.darcy import pressure_constraints
from examples.formulations.tetrahedral_darcy import (
    define_tetrahedral_darcy,
    recover_tetrahedral_darcy,
)

data = Planar3DData()
cube = TetraMesh.unit_cube()
macro = TetraMesh(cube.points, cube.cells[:1])
skeleton = TriangularSkeleton(
    macro, degree=1, face_partitions=planar_face_partitions(macro, data.material)
)
fine = fit_planar_material(macro, data.material).mesh
face = int(macro.boundary_faces[0])
with threadpool_limits(1):
    solution_definition = define_tetrahedral_darcy(
        macro,
        degree=4,
        local_meshes=(fine,),
        skeleton=skeleton,
        permeability=data.material,
        source=data.source,
        dirichlet=data.pressure,
        neumann={face: partial(data.normal_flux, normal=macro.normals[face])},
    )
    solution_system = assemble(solution_definition.problem)
    solution = recover_tetrahedral_darcy(
        solution_definition,
        solution_system,
        solution_system.solve(
            constraints=pressure_constraints(solution_definition, solution_system)
        ),
    )
    recovered = reconstruct_darcy_moments_3d(solution, degree=1)
    errors = dict(
        pressure=solution.l2_error(data.pressure),
        flux=solution.flux_l2_error(data.flux),
        reconstructed=recovered.flux_l2_error(data.flux),
    )
assert max(errors.values()) < 1e-10
errors


In [ ]:
directory = ROOT / "examples/results/planar3d"
report = json.loads((directory / "campaign.json").read_text())
assert len(report["rows"]) == 9
for row in report["rows"]:
    assert hashlib.sha256((directory / row["fields"]).read_bytes()).hexdigest() == row["fields_sha256"]
    assert max(row["pressure_l2"], row["raw_flux_l2"], row["reconstructed_flux_l2"]) < 2e-8
[(r["macro_subdivisions"], r["boundary"], r["pressure_l2"], r["raw_flux_l2"],
  r["reconstructed_flux_l2"]) for r in report["rows"]]

## Actual geometry and physical fields

The section is z=0.37. Black/white lines show original macroface intersections. Flat values retain fine-cell ownership; error color scales describe floating-point errors in these exactly represented fields.

In [ ]:
for name in ("geometry", "fields", "components"):
    display(Image(filename=str(ROOT / "docs/figures/planar3d" / f"{name}.png")))